# Quantization: GPTQ / AWQ / bitsandbytes

A refresher on **post-training weight quantization** for LLMs: shrinking 16-bit weights to 4-bit
(or 8-bit) so a model fits in less VRAM and runs faster, with minimal accuracy loss. Covers the
three names you actually meet in practice — **GPTQ**, **AWQ**, and **bitsandbytes** — what each
optimizes for, and how to drive them from `transformers`.

**Domain:** LLM Inference, Training & Optimization  ·  **recommended addition**  ·  **runnable:** yes  ·  _cross-ref [GGUF & llama.cpp](./gguf-llama-cpp.ipynb) and [QLoRA](./qlora.ipynb)_

## 1. What & Why

A 7B model in fp16 is ~14 GB of weights (2 bytes × 7B params). That doesn't fit on a 12 GB consumer
GPU, and even when it fits, memory **bandwidth** — not compute — is what caps token throughput during
generation. Quantization attacks both: store each weight in **4 bits** instead of 16 and you cut the
footprint ~4× (~3.5–4 GB for that 7B) and move ~4× fewer bytes per forward pass.

The catch: weights are continuous, low-bit integers are coarse, so quantization injects error. The
whole game is **packing weights into INT4/INT8 while preserving the model's outputs**. The three
tools below are different answers to "how do we round without wrecking accuracy?"

- **bitsandbytes (NF4 / INT8)** — *zero-effort, on-the-fly.* Pass `load_in_4bit=True` and `transformers`
  quantizes weights as it loads the fp16 checkpoint. No calibration step, no separate artifact. This is
  the backbone of **QLoRA** fine-tuning. Great for "just make it fit so I can train/experiment."
- **GPTQ** — *calibrated, one-time, accuracy-focused.* Runs a small calibration dataset through the
  model and solves a layer-wise least-squares problem (based on the Hessian / OBQ) to choose INT4 values
  that minimize output error. Produces a saved quantized checkpoint. Strong accuracy at 4-bit; the
  classic GPU-serving format.
- **AWQ (Activation-aware Weight Quantization)** — *calibrated, fast inference.* Observes that a tiny
  fraction of weight channels are "salient" (they multiply large activations) and protects those by
  per-channel scaling before rounding. Often matches or beats GPTQ accuracy with very fast fused kernels;
  the popular choice for high-throughput serving (vLLM, TGI).

**Reach for it when:** a model is too big for your VRAM, or you're memory-bandwidth bound and want more
tokens/sec. **Skip it when:** the model already fits comfortably and you need maximum accuracy, or you're
training full weights (quantize for *inference* and QLoRA, not for full fine-tuning).

## 2. Mental Model

Think of each weight tensor as a cloud of float values, and quantization as **overlaying an integer grid**
on it. You store two things: the small integers (the grid cell index) and the **scale** (how wide each cell
is). Dequantize = `int_value × scale` (+ a zero-point for asymmetric schemes).

```
fp16 weights:   -0.83  0.12  1.47  -0.05  ...   (16 bits each)
                  │      │     │      │
   pick scale = max(|w|) / 7   (INT4 signed → range -8..7)
                  ▼      ▼     ▼      ▼
INT4 codes:      -4      1     7      0          ( 4 bits each)
   store: codes + one scale per group  ──►  dequant: code × scale
```

Two levers decide how much accuracy you lose:

1. **Granularity (group size).** One scale for the whole tensor is cheap but crude — a single huge weight
   stretches the grid and everything else rounds coarsely. Instead, split each row into **groups of ~128**
   weights, each with its own scale. This is `group_size=128`, the near-universal default.
2. **Where you spend your bits.** GPTQ and AWQ both notice that *not all weights matter equally*. GPTQ
   corrects rounding error by adjusting the remaining weights (error feedback); AWQ scales up the salient
   channels so they survive rounding. bitsandbytes skips this — it just rounds with a good data type (NF4,
   shaped for normally-distributed weights).

The single sentence to remember: **quantization trades a controlled amount of rounding error for a ~4× cut
in memory and bandwidth; calibration (GPTQ/AWQ) buys back most of that error for a one-time offline cost.**

## 3. Key Concepts

| Term | What it means |
|------|---------------|
| **Scale / zero-point** | `w ≈ scale × (q − zero_point)`. *Symmetric* quantization fixes `zero_point=0` (signed ints); *asymmetric* (affine) stores a zero-point so the grid can be offset — better for skewed distributions. |
| **Per-tensor vs per-channel vs group** | Granularity of the scale. Per-tensor = one scale for everything (worst). Per-channel = one per output row. **Group-wise** (e.g. 128) = one per block of weights — the sweet spot used by GPTQ/AWQ. |
| **Weight-only quantization** | Only *weights* are stored low-bit; activations stay fp16 and weights are dequantized on the fly (or matmul'd by a fused kernel). All three tools here are weight-only. Contrast with W8A8 (also quantizing activations). |
| **Calibration data** | A few hundred sample sequences fed through the model so GPTQ/AWQ can measure activation statistics. Quality/representativeness matters; it's cheap (minutes), not training. |
| **NF4** | *NormalFloat4* — bitsandbytes' 4-bit type whose 16 levels are placed at the quantiles of a normal distribution (weights are ~Gaussian), so it spends precision where the mass is. Pairs with *double quantization* (quantizing the scales too). |
| **GPTQ** | Layer-wise post-training quantization solving an OBQ/Hessian-based least-squares to minimize output error; produces a saved INT4 checkpoint. |
| **AWQ** | Activation-aware: finds salient weight channels (large incoming activations) and applies per-channel scaling to protect them before rounding. Fast fused inference kernels. |
| **Perplexity (PPL)** | The standard accuracy proxy — run the quantized model on held-out text and compare PPL to fp16. A good 4-bit quant lands within a few hundredths to a tenth of fp16. |
| **GPTQModel / AutoAWQ / bitsandbytes** | The libraries. `transformers` integrates all three via config objects (`GPTQConfig`, `AwqConfig`, `BitsAndBytesConfig`). |

## 4. Setup

The worked examples below use **only NumPy** so they run anywhere (CPU, no downloads) — they implement the
quantization math directly so the mechanics are visible. The real-library section is *gated* behind env
checks and an `os.getenv` flag because those paths need a CUDA GPU and a model download.

```bash
# Inference with bitsandbytes 4-bit (NF4) — needs a CUDA GPU
pip install "transformers>=4.44" accelerate bitsandbytes

# Producing / loading GPTQ checkpoints
pip install gptqmodel        # maintained successor to auto-gptq
# Producing / loading AWQ checkpoints
pip install autoawq
```

bitsandbytes is **CUDA-only** for practical purposes (no usable CPU path). GPTQ/AWQ *loading* also expects a
GPU for the fused kernels. So the executable cells here stay framework-free; the GPU cells show the exact API
shape and only fire if a key/flag is set.

In [1]:
# Environment probe — what's available in THIS kernel (no downloads, no GPU needed).
import importlib.util
import sys

import numpy as np


def have(mod: str) -> str:
    return "installed" if importlib.util.find_spec(mod) else "not installed"


print(f"python        : {sys.version.split()[0]}")
print(f"numpy         : {np.__version__}")
for m in ("torch", "transformers", "bitsandbytes", "gptqmodel", "auto_gptq", "awq"):
    print(f"{m:<14}: {have(m)}")

print("\nWorked examples below are pure-NumPy and run regardless of the above.")

python        : 3.13.7
numpy         : 2.5.0
torch         : installed
transformers  : installed
bitsandbytes  : not installed
gptqmodel     : not installed
auto_gptq     : not installed
awq           : not installed

Worked examples below are pure-NumPy and run regardless of the above.


## 5. Worked Examples

### Example 1 — INT4 vs INT8 quantization, by hand

This is the entire idea in ~15 lines: pick a **scale** from the data range, round to integers, store the
small ints, then dequantize. We measure the reconstruction error and the memory saving for INT8 and INT4
symmetric (signed) quantization. Watch how going from 8 → 4 bits roughly halves the storage but multiplies
the error, because the grid is 16× coarser (16 levels vs 256).

In [2]:
rng = np.random.default_rng(0)
# Stand-in for one weight tensor: ~Gaussian, as LLM weights tend to be.
W = rng.standard_normal(4096).astype(np.float32) * 0.05


def quantize_symmetric(w, n_bits):
    """Per-tensor symmetric quant: one scale, signed integer grid, zero_point = 0."""
    qmax = 2 ** (n_bits - 1) - 1          # e.g. 127 for int8, 7 for int4
    scale = np.abs(w).max() / qmax        # width of one grid cell
    q = np.clip(np.round(w / scale), -qmax - 1, qmax)
    return q.astype(np.int8), scale


def dequantize(q, scale):
    return q.astype(np.float32) * scale


for bits in (8, 4):
    q, scale = quantize_symmetric(W, bits)
    recon = dequantize(q, scale)
    rmse = np.sqrt(np.mean((W - recon) ** 2))
    # fp16 = 2 bytes/weight; quant stores n_bits/8 bytes + tiny scale overhead.
    ratio = 16 / bits
    print(f"INT{bits}: levels={2**bits:>3}  scale={scale:.5f}  "
          f"RMSE={rmse:.5f}  rel_err={rmse / W.std():.2%}  ~{ratio:.0f}x smaller than fp16")

INT8: levels=256  scale=0.00154  RMSE=0.00044  rel_err=0.89%  ~2x smaller than fp16
INT4: levels= 16  scale=0.02785  RMSE=0.00809  rel_err=16.22%  ~4x smaller than fp16


### Example 2 — why **group size** matters (per-tensor vs group-wise)

GPTQ and AWQ don't use one scale per tensor; they use one per **group of ~128 weights**. Here's why, made
concrete: we inject a single large outlier weight (real LLM weight matrices have these) and compare INT4
error with one global scale vs per-group scales. The outlier stretches the global grid and forces every
other weight to round coarsely; group-wise quantization quarantines the damage to its own group.

In [3]:
W2 = rng.standard_normal(4096).astype(np.float32) * 0.05
W2[123] = 2.0  # one fat outlier, like the "salient" weights AWQ worries about


def quant_int4_grouped(w, group_size):
    qmax = 7
    out = np.empty_like(w)
    for start in range(0, len(w), group_size):
        g = w[start:start + group_size]
        scale = np.abs(g).max() / qmax
        q = np.clip(np.round(g / scale), -8, 7)
        out[start:start + group_size] = q * scale
    return out


def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))


per_tensor = quant_int4_grouped(W2, group_size=len(W2))   # one scale for all 4096
grouped128 = quant_int4_grouped(W2, group_size=128)       # 32 scales

# Error on the "normal" weights only (exclude the outlier itself).
mask = np.ones(len(W2), dtype=bool)
mask[123] = False
print(f"INT4 per-tensor  (1 scale)  : RMSE on normal weights = {rmse(W2[mask], per_tensor[mask]):.5f}")
print(f"INT4 group_size=128 (32 scales): RMSE on normal weights = {rmse(W2[mask], grouped128[mask]):.5f}")
print(f"\nGroup-wise is {rmse(W2[mask], per_tensor[mask]) / rmse(W2[mask], grouped128[mask]):.1f}x more "
      "accurate here — the outlier no longer poisons the whole tensor.")
print("Cost: 32 fp16 scales = 64 bytes of overhead per 4096 weights (~0.13 extra bits/weight).")

INT4 per-tensor  (1 scale)  : RMSE on normal weights = 0.04992
INT4 group_size=128 (32 scales): RMSE on normal weights = 0.01042

Group-wise is 4.8x more accurate here — the outlier no longer poisons the whole tensor.
Cost: 32 fp16 scales = 64 bytes of overhead per 4096 weights (~0.13 extra bits/weight).


### Example 3 — the real-library API shapes (gated)

In practice you don't hand-roll any of the above — you hand `transformers` a config object. The cell below
shows the exact call shape for all three tools. It's **gated behind `RUN_GPU_QUANT`** (and a GPU check) so
the notebook still executes top-to-bottom on a plain CPU; flip the env var on a CUDA box to actually run it.

In [4]:
import os

GPU = importlib.util.find_spec("torch") and __import__("torch").cuda.is_available()

if os.getenv("RUN_GPU_QUANT") and GPU:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    # --- bitsandbytes: on-the-fly NF4, no calibration, no saved artifact ---
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",          # NormalFloat4
        bnb_4bit_use_double_quant=True,     # quantize the scales too
        bnb_4bit_compute_dtype="bfloat16",  # matmul accumulation dtype
    )
    model = AutoModelForCausalLM.from_pretrained(
        "meta-llama/Llama-3.2-1B", quantization_config=bnb, device_map="auto"
    )
    print(model.get_memory_footprint() / 1e9, "GB")
else:
    print("Skipping GPU path (set RUN_GPU_QUANT=1 on a CUDA box to run). API shapes:\n")
    print(
        "# bitsandbytes (load-time, for inference + QLoRA):\n"
        "BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',\n"
        "                   bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype='bfloat16')\n\n"
        "# GPTQ (one-time calibration, then save an INT4 checkpoint):\n"
        "GPTQConfig(bits=4, group_size=128, dataset='c4', tokenizer=tok)\n"
        "  -> AutoModelForCausalLM.from_pretrained(model, quantization_config=cfg)\n"
        "  -> model.save_pretrained('model-gptq-4bit')   # reload needs no calibration\n\n"
        "# AWQ: quantize with AutoAWQ, then load the saved repo in transformers/vLLM:\n"
        "from awq import AutoAWQForCausalLM\n"
        "m = AutoAWQForCausalLM.from_pretrained(model_id)\n"
        "m.quantize(tok, quant_config={'w_bit': 4, 'q_group_size': 128, 'version': 'GEMM'})\n"
        "m.save_quantized('model-awq-4bit')"
    )

Skipping GPU path (set RUN_GPU_QUANT=1 on a CUDA box to run). API shapes:

# bitsandbytes (load-time, for inference + QLoRA):
BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                   bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype='bfloat16')

# GPTQ (one-time calibration, then save an INT4 checkpoint):
GPTQConfig(bits=4, group_size=128, dataset='c4', tokenizer=tok)
  -> AutoModelForCausalLM.from_pretrained(model, quantization_config=cfg)
  -> model.save_pretrained('model-gptq-4bit')   # reload needs no calibration

# AWQ: quantize with AutoAWQ, then load the saved repo in transformers/vLLM:
from awq import AutoAWQForCausalLM
m = AutoAWQForCausalLM.from_pretrained(model_id)
m.quantize(tok, quant_config={'w_bit': 4, 'q_group_size': 128, 'version': 'GEMM'})
m.save_quantized('model-awq-4bit')


## 6. Gotchas & Pitfalls

- **bitsandbytes is CUDA-first.** There is no production-grade CPU path; don't plan to serve NF4 on CPU
  (use GGUF/llama.cpp for that — see the cross-referenced notebook). Importing it without a GPU often errors.
- **Quantizing is not free at runtime for bitsandbytes.** It dequantizes weights on the fly each forward
  pass, so it saves memory but is *not* always faster than fp16 for compute-bound prefill. GPTQ/AWQ ship
  fused INT4 kernels that are genuinely faster for decode.
- **Group size is a real accuracy knob.** `group_size=128` is the default for a reason; `-1` (per-channel/
  per-tensor) saves a sliver of memory but can cost noticeable perplexity, especially below 4-bit.
- **Bad calibration data → bad quant.** GPTQ/AWQ calibrate on sample text; if it's off-distribution from your
  use case (e.g. code model calibrated on prose), accuracy suffers. Use representative samples.
- **4-bit is the floor for "lossless-ish."** 3-bit and 2-bit exist but degrade fast; treat sub-4-bit as
  experimental. INT8 is nearly lossless but only ~2× smaller.
- **Format ≠ engine.** A GPTQ or AWQ checkpoint must be loaded by a runtime that understands its kernels
  (transformers + the right backend, vLLM, TGI). You can't `torch.load` it as plain fp16.
- **Don't double-quantize for full training.** Quantized weights are frozen/integer; you train *adapters*
  on top (QLoRA), not the base weights. Quantization is an inference/PEFT technique, not a training one.
- **`device_map="auto"` matters.** Quantized models still need correct device placement; forgetting it can
  silently fall back to slow paths or OOM during loading.

## 7. When to Use vs Alternatives

| Option | Bits | Needs calibration | Best for | Trade-off |
|--------|------|-------------------|----------|-----------|
| **bitsandbytes (NF4/INT8)** | 4 / 8 | No | Quick fit, **QLoRA fine-tuning**, experimentation | Slower decode than fused kernels; CUDA-only |
| **GPTQ** | 4 (3/8) | Yes (one-time) | GPU serving with strong 4-bit accuracy | Calibration step; format/engine lock-in |
| **AWQ** | 4 | Yes (one-time) | **High-throughput serving** (vLLM/TGI), fast kernels | Calibration step; mainly 4-bit |
| **GGUF + llama.cpp** | 2–8 | No (k-quants) | **CPU / laptop / Apple Silicon** inference | Different ecosystem; not for training |
| **fp16 / bf16 (no quant)** | 16 | — | Max accuracy, model already fits | 4× the memory & bandwidth |
| **fp8 (H100/Ada)** | 8 | No | Newer GPUs, near-lossless, fast | Needs recent hardware |

**Rules of thumb:**
- Fine-tuning a big model on one GPU → **bitsandbytes NF4** + LoRA (= QLoRA).
- Serving on GPUs, want max throughput → **AWQ** (or GPTQ if AWQ accuracy regresses for your model).
- Serving on CPU / Mac / edge → **GGUF + llama.cpp**, not these.
- Model fits and you can't tolerate any accuracy loss → don't quantize; use bf16 (or fp8 on H100-class HW).
- Quick "does it even fit?" check → **bitsandbytes**, because there's no calibration step to wait on.

## 8. Resources

- **Transformers — Quantization overview** (bitsandbytes, GPTQ, AWQ configs side by side): <https://huggingface.co/docs/transformers/main/en/quantization/overview>
- **bitsandbytes docs** (NF4, double quant, `BitsAndBytesConfig`): <https://huggingface.co/docs/bitsandbytes/main/en/index>
- **GPTQ paper** — *Frantar et al., "GPTQ: Accurate Post-Training Quantization for Generative Pre-trained Transformers"*: <https://arxiv.org/abs/2210.17323>
- **AWQ paper** — *Lin et al., "AWQ: Activation-aware Weight Quantization for LLM Compression and Acceleration"* (MLSys 2024 best paper): <https://arxiv.org/abs/2306.00978>
- **QLoRA paper** — *Dettmers et al.*, where NF4 + double quant were introduced: <https://arxiv.org/abs/2305.14314>
- **AutoAWQ** (produce/load AWQ checkpoints): <https://github.com/casper-hansen/AutoAWQ>
- **GPTQModel** (maintained successor to AutoGPTQ): <https://github.com/ModelCloud/GPTQModel>

**Cross-refs in this library:** [QLoRA](./qlora.ipynb) (NF4 quant for fine-tuning), [GGUF & llama.cpp](./gguf-llama-cpp.ipynb) (CPU/edge quantized inference), [vLLM](./vllm.ipynb) (serves AWQ/GPTQ checkpoints).